<a href="https://colab.research.google.com/github/raghad612/sic-project-restockai/blob/raghad%2Ft06b-holiday-vs-regular/notebooks/02_eda_promo_holidays_raghad.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Mount Drive so we can read the data files
from google.colab import drive
drive.mount('/content/drive')

# Clone the repo on the right branch so "from src.config import ..." works
!git clone -b raghad/t06-eda-promo-holidays https://github.com/raghad612/sic-project-restockai
%cd sic-project-restockai

In [ ]:
import pandas as pd
from src.config import SALES_SAMPLE, STORES_OUT, HOLIDAYS_OUT

# Paths come from config, never hardcoded, so the notebook survives a folder move
sales = pd.read_parquet(SALES_SAMPLE)
stores = pd.read_parquet(STORES_OUT)
holidays = pd.read_parquet(HOLIDAYS_OUT)

# Shapes: rows and columns of each table (sales should be 1,741,225 rows)
print("sales", sales.shape)
print("stores", stores.shape)
print("holidays", holidays.shape)

# Holidays is the unverified table: look at column types and a few rows
print(holidays.dtypes)
holidays.head(10)

In [ ]:
# 1) Column names and types of sales (needed to know what we can analyse)
print(sales.dtypes)

# 2) Check onpromotion: type, number of blanks, and its values
print("onpromotion dtype:", sales["onpromotion"].dtype)
print("blanks:", sales["onpromotion"].isna().sum())
print(sales["onpromotion"].value_counts(dropna=False))

# 3) Make sure date is a real date, then split by time (never randomly)
sales["date"] = pd.to_datetime(sales["date"])
train = sales[sales["date"] <= "2017-04-30"].copy()      # all numbers come from here
late = sales[sales["date"] >= "2017-05-01"].copy()       # sanity look only, never to tune

print("train:", train.shape, train["date"].min().date(), "->", train["date"].max().date())
print("late :", late.shape, late["date"].min().date(), "->", late["date"].max().date())

In [ ]:
# Work on the training part only, and leave out the 55 negative rows
# (a filter inside this analysis only; we do not change the data, that is T7's job)
tr = train[train["unit_sales"] >= 0].copy()
print("rows used:", len(tr), "| negative rows excluded:", len(train) - len(tr))

# Guard: blanks must never silently become True, so fail loudly if any exist
assert tr["onpromotion"].isna().sum() == 0, "onpromotion has blanks: handle them before astype(bool)"

# 1) Promoted share overall: fraction of store-item-day rows with a promotion
tr["promo"] = tr["onpromotion"].astype(bool)   # plain bool, simpler for groupby
print("promoted share overall:", round(tr["promo"].mean() * 100, 2), "%")

# 2) Promoted share by store: is promotion use similar across the 12 stores?
by_store = (tr.groupby("store_nbr")["promo"].mean() * 100).round(2)
print(by_store.sort_values())

# 3) Average (and median) unit_sales, promoted vs not promoted
print(tr.groupby("promo")["unit_sales"].agg(["count", "mean", "median"]).round(2))

## Promotions (training part, up to 2017-04-30)

- Rows used: 1,527,743 (45 negative unit_sales rows excluded inside this analysis only; data not changed).
- Promoted share overall: 12.83%. By store it ranges from 10.57% (store 50) to 14.78% (store 11).
- Average unit_sales: promoted 41.13 (median 24.00) vs not promoted 28.02 (median 16.49).
- Caution: this is correlation only. Stores may promote items that already sell well, so a higher average does not prove that promotions cause more sales. Also, 20.6% of store-item-days have no row, so these averages cover only days where the item appears in the data.
- Check: onpromotion has 0 blanks (whole sample), verified before converting to bool; the code asserts it.

In [ ]:
# Keep only holidays that fall inside our training window (not 2012-2015, not after 2017-04-30)
h = holidays[(holidays["date"] >= "2015-08-01") & (holidays["date"] <= "2017-04-30")].copy()
print("holiday rows in training window:", len(h), "of", len(holidays))

# Counts by type (Holiday, Event, Additional, Bridge, Work Day, Transfer)
print(h["type"].value_counts())

# Counts by scope: National / Regional / Local
print(h["locale"].value_counts())

# Both together: which types appear at which scope
print(pd.crosstab(h["type"], h["locale"]))

# How many are flagged as transferred
print("transferred rows:", h["transferred"].sum())

## Holidays: counts (training window 2015-08-01 to 2017-04-30)

- 124 of 350 holiday rows fall in the window.
- By type: Holiday 65, Event 36, Additional 16, Transfer 5, Bridge 1, Work Day 1.
- By scope: National 74, Local 43, Regional 7.
- All 36 Events are National. Holiday rows are 20 National, 38 Local, 7 Regional.
- 5 rows have transferred = True (same count as Transfer rows; pairing checked in the next section).


In [ ]:
# 1) Rows involved in transfers: the original (transferred=True) and the Transfer rows
cols = ["date", "type", "locale", "locale_name", "description", "transferred"]
tr_rows = h[(h["transferred"]) | (h["type"] == "Transfer")].sort_values("date")
print(tr_rows[cols].to_string())

# 2) What the non-Holiday types are (read the descriptions)
other = h[h["type"].isin(["Event", "Additional", "Bridge", "Work Day"])]
print(other.groupby(["type", "description"]).size().to_string())

## Transferred holidays and other types

- Each of the 5 transferred=True rows pairs with a Transfer row for the same place, 1 to 3 days apart.
- The transferred=True row is the original date and was NOT celebrated. The Transfer row is the real celebration day.
- Examples: Batalla de Pichincha 2016-05-24 (transferred=True) -> real day 2016-05-27. Fundacion de Guayaquil 2016-07-25 -> real day 2016-07-24 (earlier). Fundacion de Cuenca 2017-04-12 -> real day 2017-04-13.
- Events are not days off: 31 Terremoto Manabi rows (earthquake + 30 days), Black Friday 2, Cyber Monday 2, Dia de la Madre 1.
- Additional (16) are days around Christmas/New Year and a few others; Bridge 1; Work Day 1.

In [ ]:
# The 12 stores in our sample, with the city and state used for matching
store_ids = sorted(sales["store_nbr"].unique())
s12 = stores[stores["store_nbr"].isin(store_ids)][["store_nbr", "city", "state"]]
print(s12.to_string())

# Rows that mean "a celebrated day": drop Events, Work Day and transferred=True originals
real = h[(~h["transferred"]) & (~h["type"].isin(["Event", "Work Day"]))]

# National: every holiday date applies to all 12 stores (cross join on a dummy key)
nat = (real[real["locale"] == "National"][["date"]].drop_duplicates()
       .assign(k=1).merge(s12.assign(k=1), on="k").drop(columns="k"))

# Regional: holiday locale_name must equal the store's STATE
reg = real[real["locale"] == "Regional"].merge(s12, left_on="locale_name", right_on="state")

# Local: holiday locale_name must equal the store's CITY
loc = real[real["locale"] == "Local"].merge(s12, left_on="locale_name", right_on="city")

# Count distinct store-days each scope touches, out of all possible store-days in training
n_days = (pd.Timestamp("2017-04-30") - pd.Timestamp("2015-08-01")).days + 1
total = n_days * len(store_ids)
for name, df in [("national", nat), ("regional", reg), ("local", loc)]:
    n = df[["date", "store_nbr"]].drop_duplicates().shape[0]
    print(f"{name}: {n} store-days ({n/total*100:.2f}% of {total})")

# Which cities/states actually matched (check against the s12 table above)
print(loc.groupby("city").size())
print(reg.groupby("state").size())

In [ ]:
# 1) The 7 regional rows in the window: which provinces, which dates, transferred or not
print(h[h["locale"] == "Regional"][["date", "type", "locale_name", "description", "transferred"]].to_string())

# 2) Our states vs regional names: any overlap? Repr shows hidden spaces or accents
print("our states:", sorted(s12["state"].unique()))
print("regional names:", sorted(h[h["locale"] == "Regional"]["locale_name"].unique()))

# 3) Same check for the two cities with no local match (Daule, Babahoyo)
print(holidays[holidays["locale_name"].isin(["Daule", "Babahoyo"])][["date", "locale", "locale_name", "description"]].to_string())

## Matching holidays to stores (12 stores, 2015-08-01 to 2017-04-30)

Rule: National applies to all 12 stores; Regional matches the store's state; Local matches the store's city. Celebrated days only: Events, Work Day and transferred=True originals excluded.

- National: 408 store-days (5.32% of 7,668), from 34 distinct dates x 12 stores.
- Regional: 0 store-days. The 7 regional rows belong to Santo Domingo de los Tsachilas, Santa Elena, Cotopaxi and Imbabura; none is among our 6 states (checked: names do not overlap).
- Local: 22 store-days (0.29%), matched in 8 of our 10 cities (Quito, Ambato, Guayaquil, Libertad, Cuenca, Quevedo, Cayambe, Machala). Daule and Babahoyo have no holiday rows in the table.
- The 24 matched local rows collapse to 22 distinct store-days (2 duplicates), so T8 must deduplicate before merging.
- Pitfall avoided: applying a local or regional holiday to every store.

In [ ]:
import matplotlib.pyplot as plt

# 0) Check the two holiday rows exist in our table (stop with a clear message if not)
nat_row = h[(h["description"] == "Navidad") & (h["date"] == "2016-12-25")]
loc_row = h[(h["description"] == "Fundacion de Quito") & (h["date"] == "2016-12-06")]
assert len(nat_row) == 1 and len(loc_row) == 1, "holiday row not found, paste h[h['description'].str.contains('Navidad|Quito')]"
print(nat_row[["date", "type", "locale", "locale_name"]].to_string())
print(loc_row[["date", "type", "locale", "locale_name"]].to_string())

def window_units(center, store_list=None, days=7):
    """Total units per day, from -days to +days around the holiday.
    Why total units: one simple number per day. Why reindex: a day with no rows
    (stores closed) must show as a gap, not silently disappear."""
    c = pd.Timestamp(center)
    d = tr[(tr["date"] >= c - pd.Timedelta(days=days)) & (tr["date"] <= c + pd.Timedelta(days=days))]
    if store_list is not None:
        d = d[d["store_nbr"].isin(store_list)]
    s = d.groupby("date")["unit_sales"].sum()
    s = s.reindex(pd.date_range(c - pd.Timedelta(days=days), c + pd.Timedelta(days=days)))
    s.index = (s.index - c).days            # x axis = days from the holiday
    return s

nat_s = window_units("2016-12-25")                      # national: all 12 stores
loc_s = window_units("2016-12-06", store_list=[17, 44])  # local: Quito stores only
print(pd.DataFrame({"national_Navidad": nat_s, "local_Quito": loc_s}).round(0))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for a, s, t in [(ax[0], nat_s, "National: Navidad 2016-12-25 (12 stores)"),
                (ax[1], loc_s, "Local: Fundacion de Quito 2016-12-06 (stores 17, 44)")]:
    a.plot(s.index, s.values, marker="o")
    a.axvline(0, color="red", linestyle="--")       # the holiday itself
    a.set_title(t); a.set_xlabel("days from holiday"); a.set_ylabel("total units")
plt.tight_layout(); plt.show()

## Sales around two holidays (training part, 7 days before to 7 days after)

National: Navidad 2016-12-25 (12 stores). Local: Fundacion de Quito 2016-12-06 (stores 17 and 44). Regional skipped (0 store-days touch our stores).

- National: no sales rows exist for any of the 12 stores on Dec 25 and Jan 1. Closure is a possible reason but is not verified, so we call these days with no recorded sales. Dec 24 is high (99,616 units) and Dec 26-29 are lower (71,413 to 86,289).
- Local: Dec 6 sold 22,938 units vs 16,057 and 15,946 on the same weekday one week before and after (about +43%).
- The weekly pattern is strong: Sundays are the highest days in the Quito window (29,770 on Dec 4, 27,505 on Dec 11).
- Pitfalls: compare a holiday with the same weekday, not the neighbouring day. The national window mixes Christmas, the Navidad-1 to -4 days and New Year. Two holidays in one year are examples, not proof.

In [ ]:
# Verify the weekdays I assumed (Dec 6 should be Tuesday, Dec 4 and Dec 11 Sundays, Dec 25 Sunday)
for d in ["2016-12-04", "2016-12-06", "2016-12-11", "2016-12-18", "2016-12-25", "2017-01-01"]:
    print(d, pd.Timestamp(d).day_name())

In [ ]:
# Sanity only: same promotion numbers on the last 3 months. No decision is taken from this.
lt = late[late["unit_sales"] >= 0].copy()          # same negative-row filter as in training
lt["promo"] = lt["onpromotion"].astype(bool)
print("rows:", len(lt), "| negative rows excluded:", len(late) - len(lt))
print("promoted share overall:", round(lt["promo"].mean() * 100, 2), "%")
print(lt.groupby("promo")["unit_sales"].agg(["count", "mean", "median"]).round(2))
print((lt.groupby("store_nbr")["promo"].mean() * 100).round(2).sort_values())

In [ ]:
import numpy as np

# 1) One number per store per day: total units over the 250 items (training part, negatives already out)
sd = tr.groupby(["store_nbr", "date"])["unit_sales"].sum().rename("units").reset_index()

# 2) Drop the earthquake window: it lifts sales for 30 days and would distort the baseline
quake = h[h["description"].str.startswith("Terremoto")]["date"]
sd = sd[~sd["date"].between(quake.min(), quake.max())]

# 3) Mark celebrated holiday store-days (national + local, from Step 4; regional is empty)
hol = (pd.concat([nat[["date", "store_nbr"]].assign(scope="national"),
                  loc[["date", "store_nbr"]].assign(scope="local")])
       .drop_duplicates(["date", "store_nbr"]))      # a store-day counts once
n0 = len(sd)
sd = sd.merge(hol.assign(is_hol=1), on=["date", "store_nbr"], how="left")
assert len(sd) == n0, "merge duplicated rows"         # Nour's point: one record per store-date
sd["is_hol"] = sd["is_hol"].fillna(0).astype(int)

# 4) Baseline per holiday store-day: same store, same weekday, non-holiday, within +-4 weeks
lookup = sd.set_index(["store_nbr", "date"])
rows = []
for _, r in sd[sd["is_hol"] == 1].iterrows():
    vals = []
    for k in (-28, -21, -14, -7, 7, 14, 21, 28):
        key = (r["store_nbr"], r["date"] + pd.Timedelta(days=k))
        if key in lookup.index and lookup.loc[key, "is_hol"] == 0:
            vals.append(lookup.loc[key, "units"])
    if len(vals) >= 3:                                 # need enough comparison days
        rows.append((r["date"], r["store_nbr"], r["scope"], r["units"], np.mean(vals)))
res = pd.DataFrame(rows, columns=["date", "store_nbr", "scope", "units", "baseline"])
res["ratio"] = res["units"] / res["baseline"]

# 5) Results: by scope, then national by date (does one date drive the average?)
print("holiday store-days compared:", len(res), "of", int(sd["is_hol"].sum()))
print(res.groupby("scope")["ratio"].agg(["count", "mean", "median"]).round(3))
print(res[res["scope"] == "national"].groupby("date")["ratio"].mean().round(2).to_string())

In [ ]:
# National dates with their type; each date must have exactly one type, or the merge would duplicate rows
typ = real[real["locale"] == "National"][["date", "type"]].drop_duplicates()
assert typ["date"].is_unique, "a date has two types: paste typ[typ.duplicated('date', keep=False)]"

nat_res = res[res["scope"] == "national"].merge(typ, on="date", how="left")
assert nat_res["type"].notna().all()

# Two groups: shopping days (Additional, Bridge) vs days off (Holiday, Transfer)
nat_res["group"] = np.where(nat_res["type"].isin(["Additional", "Bridge"]),
                            "shopping days (Additional/Bridge)", "days off (Holiday/Transfer)")
print(nat_res.groupby("group").agg(store_days=("ratio", "count"), dates=("date", "nunique"),
                                   mean=("ratio", "mean"), median=("ratio", "median")).round(3))

## Holiday vs regular days (same weekday)

Method: for each holiday store-day (national + local, Step 4 rules), baseline = mean total units of the same store on the same weekday, non-holiday days, within +-4 weeks (at least 3 comparison days). Ratio = holiday / baseline. Total units per store-day over the 250 items, training part, earthquake window excluded.

- 369 holiday store-days compared (348 national, 21 local).
- National overall: mean 1.226, median 1.175. It mixes two groups:
  - Days off (Holiday/Transfer): 192 store-days, 16 dates, mean 1.098, median 1.076.
  - Shopping days (Additional/Bridge): 156 store-days, 13 dates, mean 1.384, median 1.303.
- Local: 21 store-days, mean 0.955, median 0.92: no clear effect. The Quito example (+43%) was one holiday in one year.
- Per national date the ratio ranges from 0.76 to 1.77. Lowest: 2016-02-08 (0.76), 2017-02-27 (0.84), 2016-02-09 and 2016-03-25 (0.94).
- Pitfalls: paydays and the Christmas season are not fully removed by the baseline; few dates per group; correlation, not cause; Dec 25 and Jan 1 have no rows and cannot be compared.
- Left for later: effect of promotions by product family.

## How T8 should encode promotions and holidays (decisions from T6)

1. Promotions: use onpromotion as a 0/1 column (boolean, 0 blanks). Promoted share is 12.83% overall, 10.57% to 14.78% by store.
   Pitfall: promoted rows sell more on average (41.13 vs 28.02), but that is partly item popularity. Keep item-level features next to the flag, or the model credits promotions for what is really popularity.

2. National holidays: build TWO 0/1 flags, same for all 12 stores: "day off" (Holiday and Transfer rows, 16 dates, about +10%) and "shopping day" (Additional and Bridge rows, 13 dates, about +38%).
   Pitfall: one merged flag would blur a +10% effect with a +38% effect. Build from celebrated days only (no Events, Work Day, transferred=True).

3. Regional holidays: no regional holiday touches our 12 stores (0 store-days), so do not build a regional flag for v1.
   Pitfall: a column that is always 0 adds nothing and clutters feature importance.

4. Local holidays: one 0/1 flag per store-day, matched on city (22 store-days). Low priority: 21 compared store-days show no clear effect (mean ratio 0.955).
   Pitfall: applying a local holiday to every store, and expecting the model to learn from 0.29% of store-days.

5. Transferred holidays: flag the date of the Transfer row (the real day) and ignore the date of the transferred=True row.
   Pitfall: flagging the original date, when nothing was celebrated.

6. Events: keep them out of the holiday flags. Terremoto Manabi (31 days) is a one-off shock, so give it its own flag. Black Friday and Cyber Monday can share a shopping-event flag.
   Pitfall: mixing the earthquake into the holiday flag. It will not repeat, so do not expect it to generalise.

7. Additional and Bridge rows are NOT merged with days off any more: they get their own "shopping day" flag (see point 2).
   Pitfall: counting them as holidays hides the strongest effect we found.

8. Build every flag from the date, locale and place columns, never from Ecuadorian holiday names.
   Pitfall: names like "Batalla de Pichincha" would break the method in another country. With generic flags, a Lebanese holiday calendar can replace this one.
9. Days to next / since last holiday: candidate features, with "days to Christmas" the most promising, since the shopping days cluster there (ratios 1.19 to 1.77 on Dec 21-24, Dec 31, Jan 2). Tentative: 13 shopping dates over two years.
   Pitfall: do not keep them without a check on the time split.

10. No recorded sales: on Dec 25 and Jan 1 no store has any row. Closure is not verified, so treat them as days with no recorded sales and keep them as missing, never as zero sales.
   Pitfall: filling them with 0 would teach the model that holidays mean zero demand, which we cannot confirm. This ties to the 20.6% of store-item-days with no row, which belongs to T7/T8.

11. Before merging holiday flags with sales, keep one record per date (national) and one per store-date (local). In T6, 24 matched local rows were only 22 distinct store-days.
   Pitfall: a many-to-one merge silently duplicates sales rows and inflates every total.